# 01. 텐서 기초

PyTorch에서 입력·가중치·출력은 전부 **텐서(Tensor)** 다. NumPy 배열과 거의 같고, 여기에 두 가지가 추가된다.

1. GPU에서 연산할 수 있다.
2. 연산 기록을 남겨 **자동으로 미분**할 수 있다 (→ 02).

순서: 만들기 → 속성 → 인덱싱 → 연산과 `dim` → 브로드캐스팅 → 모양 바꾸기 → 행렬곱

In [1]:
import torch
import numpy as np

print(torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

2.15.0+cpu
device: cpu


## 1. 텐서 만들기

In [2]:
a = torch.tensor([1, 2, 3])              # 정수 리스트 → int64
b = torch.tensor([[1., 2.], [3., 4.]])   # 실수 리스트 → float32
print(a, a.dtype)
print(b, b.dtype)

print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.arange(0, 10, 2))            # [0, 2, 4, 6, 8]
print(torch.linspace(0, 1, 5))           # 0~1을 5등분

torch.manual_seed(0)                     # 난수 고정 (재현성)
print(torch.rand(2, 3))                  # 균등분포 [0, 1)
print(torch.randn(2, 3))                 # 표준정규분포 N(0, 1)
print(torch.randint(0, 10, (2, 3)))      # 0~9 정수

tensor([1, 2, 3]) torch.int64
tensor([[1., 2.],
        [3., 4.]]) torch.float32
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([0, 2, 4, 6, 8])
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]])
tensor([[ 1.2645, -0.6874,  0.1604],
        [-0.6065, -0.7831,  1.0622]])
tensor([[4, 3, 6],
        [9, 1, 4]])


## 2. 속성: `shape`, `dtype`, `device`

- 모델 입력, 가중치 → `float32`
- 토큰 ID, 분류 정답 라벨 → `int64` (= `torch.long`). `nn.Embedding`과 `CrossEntropyLoss`가 이것을 요구한다.

In [3]:
x = torch.randn(4, 3, 2)
print("shape :", x.shape, "| size(1):", x.size(1), "| shape[-1]:", x.shape[-1])
print("ndim  :", x.ndim)
print("numel :", x.numel())      # 원소 개수 4*3*2
print("dtype :", x.dtype)
print("device:", x.device)

ids = torch.tensor([3, 1, 4])
print(ids.dtype, ids.float().dtype)   # dtype 변환
x = x.to(device)                      # GPU가 있으면 GPU로

shape : torch.Size([4, 3, 2]) | size(1): 3 | shape[-1]: 2
ndim  : 3
numel : 24
dtype : torch.float32
device: cpu
torch.int64 torch.float32


## 3. 인덱싱과 슬라이싱 (NumPy와 동일)

In [4]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("x[0]       :", x[0])          # 첫 행
print("x[:, 1]    :", x[:, 1])       # 둘째 열
print("x[1:, :2]  :\n", x[1:, :2])
print("x[..., -1] :", x[..., -1])    # 마지막 차원의 마지막 원소 (차원 수 무관)
print("x[x > 5]   :", x[x > 5])      # 불리언 마스크
print("x[[0, 2]]  :\n", x[[0, 2]])   # 정수 리스트로 행 여러 개 선택 ← nn.Embedding이 하는 일

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
x[0]       : tensor([0, 1, 2, 3])
x[:, 1]    : tensor([1, 5, 9])
x[1:, :2]  :
 tensor([[4, 5],
        [8, 9]])
x[..., -1] : tensor([ 3,  7, 11])
x[x > 5]   : tensor([ 6,  7,  8,  9, 10, 11])
x[[0, 2]]  :
 tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])


## 4. 연산과 `dim`

- `+ - * /` 는 **원소별** 연산이다. 행렬곱은 `@`.
- `sum / mean / max / argmax / softmax (dim=k)` 는 **k번 차원을 따라 계산하고 그 차원을 없앤다**. `keepdim=True`면 그 차원을 크기 1로 남긴다.

`(2, 3)` 텐서에서 `dim=0`은 "행들을 따라" (세로로) 합쳐 `(3,)`, `dim=1`은 "열들을 따라" (가로로) 합쳐 `(2,)` 가 된다.

In [5]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])       # (2, 3)
print(x * 10)
print(x * x)                            # 원소별 곱
print("sum()      :", x.sum())
print("sum(dim=0) :", x.sum(dim=0))     # (3,)
print("sum(dim=1) :", x.sum(dim=1))     # (2,)
print("mean(dim=-1, keepdim=True).shape:", x.mean(dim=-1, keepdim=True).shape)   # (2, 1)
print("argmax(dim=1):", x.argmax(dim=1))
print("softmax(dim=1):\n", torch.softmax(x, dim=1))   # 각 행의 합이 1

tensor([[10., 20., 30.],
        [40., 50., 60.]])
tensor([[ 1.,  4.,  9.],
        [16., 25., 36.]])
sum()      : tensor(21.)
sum(dim=0) : tensor([5., 7., 9.])
sum(dim=1) : tensor([ 6., 15.])
mean(dim=-1, keepdim=True).shape: torch.Size([2, 1])
argmax(dim=1): tensor([2, 2])
softmax(dim=1):
 tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])


## 5. 브로드캐스팅

모양이 다른 텐서끼리 연산할 때 **뒤쪽 차원부터 맞춰 보고**, 크기가 1이거나 없는 차원은 자동으로 늘려서 계산한다.

In [6]:
x = torch.ones(2, 3)
print(x + torch.tensor([10., 20., 30.]))      # (2,3) + (3,)  → 각 행에 더함
print(x + torch.tensor([[100.], [200.]]))     # (2,3) + (2,1) → 각 열에 더함

# 실전 1: 토큰 임베딩 (B, T, D) + 위치 임베딩 (T, D)
tok = torch.zeros(8, 5, 16)
pos = torch.randn(5, 16)
print((tok + pos).shape)                      # (8, 5, 16)

# 실전 2: 마지막 차원 기준 정규화 (LayerNorm의 핵심). keepdim으로 (2,1)을 만들어야 브로드캐스팅 가능
h = torch.randn(2, 4)
h_norm = (h - h.mean(dim=-1, keepdim=True)) / h.std(dim=-1, keepdim=True)
print(h_norm.mean(dim=-1), h_norm.std(dim=-1))

tensor([[11., 21., 31.],
        [11., 21., 31.]])
tensor([[101., 101., 101.],
        [201., 201., 201.]])
torch.Size([8, 5, 16])
tensor([-9.3132e-10,  2.4214e-08]) tensor([1.0000, 1.0000])


## 6. 모양 바꾸기

| 함수 | 하는 일 |
| --- | --- |
| `reshape` / `view` | 원소 순서 그대로 모양만 바꿈 (`-1`은 자동 계산) |
| `transpose(i, j)` | 두 차원 교환 |
| `permute(...)` | 여러 차원 순서를 한 번에 재배치 |
| `unsqueeze(k)` / `squeeze(k)` | 크기 1인 차원 추가 / 제거 |
| `flatten(s, e)` | s~e 차원을 하나로 합침 |
| `cat` / `stack` | 기존 차원으로 이어 붙이기 / 새 차원을 만들어 쌓기 |

In [7]:
x = torch.arange(24)
print(x.view(2, 3, 4).shape)
print(x.view(2, -1).shape)                         # (2, 12)

y = x.view(2, 3, 4)
print("transpose(1, 2):", y.transpose(1, 2).shape)  # (2, 4, 3)
print("permute(2, 0, 1):", y.permute(2, 0, 1).shape)  # (4, 2, 3)
print("unsqueeze(0)   :", y.unsqueeze(0).shape)     # (1, 2, 3, 4)
print("flatten(0, 1)  :", y.flatten(0, 1).shape)    # (6, 4)

a, b = torch.zeros(2, 3), torch.ones(2, 3)
print("cat dim=0  :", torch.cat([a, b], dim=0).shape)    # (4, 3)
print("cat dim=1  :", torch.cat([a, b], dim=1).shape)    # (2, 6)
print("stack dim=0:", torch.stack([a, b], dim=0).shape)  # (2, 2, 3)

torch.Size([2, 3, 4])
torch.Size([2, 12])
transpose(1, 2): torch.Size([2, 4, 3])
permute(2, 0, 1): torch.Size([4, 2, 3])
unsqueeze(0)   : torch.Size([1, 2, 3, 4])
flatten(0, 1)  : torch.Size([6, 4])
cat dim=0  : torch.Size([4, 3])
cat dim=1  : torch.Size([2, 6])
stack dim=0: torch.Size([2, 2, 3])


`transpose` 뒤에는 메모리 배치가 연속적이지 않아서 `view`가 실패할 수 있다. 그럴 땐 `.contiguous().view(...)` 나 `reshape`를 쓴다. (Multi-Head Attention 코드에 나오는 `.contiguous()`가 이것.)

In [8]:
t = y.transpose(1, 2)
print("contiguous?", t.is_contiguous())
try:
    t.view(2, 12)
except RuntimeError as e:
    print("view 실패:", str(e)[:70], "...")
print(t.contiguous().view(2, 12).shape, t.reshape(2, 12).shape)

contiguous? False
view 실패: view size is not compatible with input tensor's size and stride (at le ...
torch.Size([2, 12]) torch.Size([2, 12])


## 7. 행렬곱 `@`

**마지막 두 차원**으로 행렬곱을 하고, 앞쪽 차원은 배치로 취급한다.

In [9]:
W = torch.randn(3, 5)
x = torch.randn(3)
print((x @ W).shape)                  # (3,) @ (3,5) → (5,)

X = torch.randn(10, 3)                # 샘플 10개
print((X @ W).shape)                  # (10, 5)  ← nn.Linear가 하는 일

Q = torch.randn(2, 6, 8)              # (배치, 토큰, 차원)
K = torch.randn(2, 6, 8)
print((Q @ K.transpose(1, 2)).shape)  # (2, 6, 6) ← 어텐션 점수 행렬

torch.Size([5])
torch.Size([10, 5])
torch.Size([2, 6, 6])


## 8. NumPy / 파이썬 값과 오가기

In [10]:
n = np.array([1.0, 2.0])
t = torch.from_numpy(n)
print(t, t.dtype)                     # float64! 모델에 넣을 땐 .float()
print(torch.tensor([1., 2.]).numpy())
print(torch.tensor(3.14).item())      # 스칼라 텐서 → 파이썬 숫자 (loss.item()으로 자주 씀)
print(torch.tensor([[1, 2]]).tolist())

tensor([1., 2.], dtype=torch.float64) torch.float64
[1. 2.]
3.140000104904175
[[1, 2]]
